# SOC Analyst — Virgil
### Jev (decision engine) + LLM investigator (Gemini / OpenAI-compatible / Claude)

```
ALERT
  │
  ▼
PASS 1 · Jev scores the raw alert (8 questions, one API call)
  │
  ▼
CONFIDENCE GATE ── not confident ──► INVESTIGATOR · LLM retrieves correlated telemetry
  │                                            │
  │◄──────────── enriched state ◄──────────────┘
  ▼
PASS 2 · Jev re-scores with evidence
  │
  ▼
RESPONSE · automated actions, or human review when under threshold
```

**How to read a trace:** every alert prints its pass-1 scores, *why* the gate routed it, what the
investigator retrieved, pass-2 scores with deltas vs pass 1, the actions taken or held (with
reasons), and a plain-English **assessment** written by the investigator.

**Key design points**
- Pass 1 only sees the *raw* alert. The `enrichment` / `correlated_activity` sections of the demo
  states play the role of backend telemetry — only the investigator may retrieve them, and pass 2
  is the only pass that ever sees them.
- The gate and action policy are plain thresholds (config cell) — deterministic and auditable.
- The investigator is provider-agnostic: Google Gemini, any OpenAI-compatible endpoint
  (OpenAI, Minimax, …), or Anthropic Claude — one flag in the config cell.
- **No API keys?** The notebook still runs end-to-end: Jev falls back to clearly-labeled mock
  scores and the investigator to a deterministic heuristic. Set the keys for the real demo.

In [ ]:
# ============================================================
# 1. SETUP
# ============================================================
# Hosted services only - no local models to install.
#
#   Jev (decision engine)       early-access key: https://typesafe.ai
#       export TYPESAFE_API_KEY="..."
#   Investigator (pick one)
#       Gemini:   https://aistudio.google.com/apikey   -> export GEMINI_API_KEY="..."
#                 (%pip install -q google-genai)
#       OpenAI-compatible (OpenAI / Minimax / ...):     -> export OPENAI_API_KEY="..."
#                 (no SDK needed - plain REST)
#       Claude:   https://console.anthropic.com        -> export ANTHROPIC_API_KEY="..."
#                 (no SDK needed - plain REST)
#
# Or paste keys directly into the config cell below.
print("See comments above - hosted services only, nothing to install here.")

In [ ]:
# ============================================================
# 2. CONFIG — every knob in one place
# ============================================================

# --- Credentials (or set the env vars before launching Jupyter) ---
TYPESAFE_API_KEY = None        # Jev decision engine      (https://typesafe.ai)
GEMINI_API_KEY   = None        # Google Gemini            (https://aistudio.google.com/apikey)
COMPAT_API_KEY   = None        # OpenAI / Minimax / ...   (or OPENAI_API_KEY / MINIMAX_API_KEY env)
CLAUDE_API_KEY   = None        # Anthropic Claude         (or ANTHROPIC_API_KEY env)

# --- Models ---
JEV_MODEL     = "jev-latest"
GEMINI_MODEL  = "gemini-3.8-flash"
COMPAT_MODEL  = "gpt-5.6-luna"        # any chat model id your endpoint serves (e.g. GLM 5.3 Flash)
CLAUDE_MODEL  = "claude-haiku-4-5"

# --- Investigator provider: "gemini" | "openai_compatible" | "claude" ---
INVESTIGATOR_PROVIDER = "gemini"
# OpenAI-compatible endpoint base URL (no trailing slash):
#   OpenAI:  https://api.openai.com/v1        Minimax: https://api.minimax.io/v1        ZAI: https://api.z.ai/v1
COMPAT_BASE_URL = "https://api.openai.com/v1"

# --- Confidence gate: when is the machine confident enough to act? ---
TP_ACT_THRESHOLD      = 0.95   # auto-response requires P(true_positive) >= this
FP_CLOSE_THRESHOLD    = 0.10   # P(true_positive) <= this -> auto-close as benign
INVESTIGATE_THRESHOLD = 0.35   # P(needs investigation) >= this -> escalate to investigator

# --- Response policy: per-action probability bars, tiered by impact ---
ACTION_POLICY = {
    "increase_monitoring": {"threshold": 0.60, "impact": "low"},     # reversible
    "preserve_evidence":   {"threshold": 0.70, "impact": "low"},     # reversible
    "block_indicator":     {"threshold": 0.85, "impact": "medium"},
    "isolate_endpoint":    {"threshold": 0.95, "impact": "high"},
    "disable_account":     {"threshold": 0.95, "impact": "high"},
}
# High-impact actions on critical assets always need human sign-off
HUMAN_APPROVAL_ON_CRITICAL = True
CRITICAL_HOST_LEVELS = {"critical"}

# --- Demo plumbing ---
MOCK_JEV = None                 # None = mock iff no key | True = force mock | False = require key
SHOW_RICH_TABLES = False        # also render pandas tables alongside the text trace
HIDDEN_UNTIL_INVESTIGATION = ("enrichment", "correlated_activity")  # telemetry sections

import os, re, json, time, textwrap, urllib.request, urllib.error
from copy import deepcopy
import pandas as pd
from IPython.display import display

In [ ]:
# ============================================================
# 3. VERIFY CONNECTIONS — go/no-go before running the demo
# ============================================================
# Confirms the Jev key and the configured investigator provider key, each with a
# minimal live call.

def _mask(v):
    return f"set (...{v[-4:]}, {len(v)} chars)" if v else "MISSING"

def _post_json(url, body, headers, timeout=30):
    req = urllib.request.Request(url, data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json", **headers})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())

# --- Jev ---
_jev = TYPESAFE_API_KEY or os.environ.get("TYPESAFE_API_KEY")
print(f"TYPESAFE_API_KEY : {_mask(_jev)}")
if _jev:
    try:
        _ans = _post_json("https://api.typesafe.ai/v1/systemone",
                          {"model": JEV_MODEL, "state": {"alert": {"rule": "connectivity test"}},
                           "questions": {"ping": {"type": "noul",
                                                  "instructions": "Is this a connectivity test?"}}},
                          {"Authorization": f"Bearer {_jev}"})["answers"]["ping"]
        print(f"Jev API    : OK (live call returned {_ans})")
    except urllib.error.HTTPError as e:
        print(f"Jev API    : HTTP {e.code} — {e.read().decode()[:200]}  (401/403 = bad key)")
    except Exception as e:
        print(f"Jev API    : FAILED — {e}")
else:
    print("Jev API    : no key -> pipeline will use MOCK decision scores")

# --- Investigator provider ---
print(f"\nInvestigator provider: {INVESTIGATOR_PROVIDER}")
if INVESTIGATOR_PROVIDER == "gemini":
    _k = GEMINI_API_KEY or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    print(f"GEMINI_API_KEY   : {_mask(_k)}")
    if _k:
        try:
            from google import genai
            _r = genai.Client(api_key=_k).models.generate_content(
                model=GEMINI_MODEL, contents="Reply with exactly: ok")
            print(f"Gemini API : OK ({GEMINI_MODEL} returned {_r.text.strip()!r})")
        except Exception as e:
            print(f"Gemini API : FAILED — {e}")
elif INVESTIGATOR_PROVIDER == "openai_compatible":
    _k = COMPAT_API_KEY or os.environ.get("OPENAI_API_KEY") or os.environ.get("MINIMAX_API_KEY")
    print(f"COMPAT_API_KEY   : {_mask(_k)}   endpoint: {COMPAT_BASE_URL}")
    if _k:
        try:
            _r = _post_json(f"{COMPAT_BASE_URL}/chat/completions",
                            {"model": COMPAT_MODEL,
                             "messages": [{"role": "user", "content": "Reply with exactly: ok"}]},
                            {"Authorization": f"Bearer {_k}"})
            print(f"Compat API : OK ({COMPAT_MODEL} returned "
                  f"{_r['choices'][0]['message']['content'].strip()!r})")
        except urllib.error.HTTPError as e:
            print(f"Compat API : HTTP {e.code} — {e.read().decode()[:200]}")
        except Exception as e:
            print(f"Compat API : FAILED — {e}")
elif INVESTIGATOR_PROVIDER == "claude":
    _k = CLAUDE_API_KEY or os.environ.get("ANTHROPIC_API_KEY")
    print(f"CLAUDE_API_KEY   : {_mask(_k)}")
    if _k:
        try:
            _r = _post_json("https://api.anthropic.com/v1/messages",
                            {"model": CLAUDE_MODEL, "max_tokens": 16,
                             "messages": [{"role": "user", "content": "Reply with exactly: ok"}]},
                            {"x-api-key": _k, "anthropic-version": "2023-06-01"})
            print(f"Claude API : OK ({CLAUDE_MODEL} returned {_r['content'][0]['text'].strip()!r})")
        except urllib.error.HTTPError as e:
            print(f"Claude API : HTTP {e.code} — {e.read().decode()[:200]}")
        except Exception as e:
            print(f"Claude API : FAILED — {e}")

In [ ]:
# ============================================================
# 4. DATA — the 8 triage questions and 40 demo alert states
# ============================================================
# Loads questions.json / demo_states.json next to the notebook (either spelling);
# falls back to embedded copies so the notebook always runs.
# If auto-detect fails, set DATA_DIR to the folder containing the files.
DATA_DIR = None

from pathlib import Path
import json as _json

def _find(*names):
    for d in ([Path(DATA_DIR)] if DATA_DIR else []) + [Path.cwd()]:
        for n in names:
            if (d / n).exists():
                return d / n
    return None

_q = _find("questions.json")
_s = _find("demo_states.json", "demo-states.json", "virgil-states.json", "Demo-States.txt")
if _q and _s:
    QUESTIONS = _json.loads(_q.read_text(encoding="utf-8"))
    STATES = _json.loads(_s.read_text(encoding="utf-8"))
    print(f"Loaded {len(QUESTIONS)} questions and {len(STATES)} alert states from {_q.parent}")
else:
    print(f"JSON files not found")

In [ ]:
# ============================================================
# 5. SHARED HELPERS
# ============================================================

def strip_for_pass1(state):
    """Raw alert view: drops telemetry sections until the investigator retrieves them."""
    return {k: v for k, v in state.items() if k not in HIDDEN_UNTIL_INVESTIGATION}


def deep_merge(base, extra):
    """Recursively merge retrieved evidence into the alert state."""
    for k, v in extra.items():
        if isinstance(v, dict) and isinstance(base.get(k), dict):
            deep_merge(base[k], v)
        else:
            base[k] = v
    return base


def fmt_scores(scores, prev=None, indent=4):
    """Aligned score block; with `prev` (pass-1 scores) shows per-question deltas."""
    pad, width = " " * indent, max(len(q) for q in scores)
    for qid, p in scores.items():
        line = f"{pad}{qid.replace('_', ' '):<{width}}  {p:>5.2f}"
        if prev is not None:
            d = p - prev[qid]
            line += f"   (was {prev[qid]:.2f}, {'+' if d >= 0 else ''}{d:.2f})"
        print(line)


def show_scores(scores, title="", prev=None):
    if title:
        print(title)
    fmt_scores(scores, prev)
    if SHOW_RICH_TABLES:
        display(pd.DataFrame({"P(true)": scores}).T.round(3))

In [ ]:
# ============================================================
# 6. DECISION ENGINE — Jev (TypeSafe hosted System One)
# ============================================================
# One POST scores all 8 noul questions in parallel -> {question_id: P(true)}.
# noul answers carry just the probability (no separate confidence field).
# With no key, MOCK mode (bottom of this cell) keeps the demo runnable offline.

JEV_ENDPOINT = "https://api.typesafe.ai/v1/systemone"
_mock_notice_shown = False

def jev_decisions(state, questions=QUESTIONS, timeout=30, retries=2):
    """Score one alert state against all 8 questions. Returns {qid: P(true)}."""
    global _mock_notice_shown
    key = TYPESAFE_API_KEY or os.environ.get("TYPESAFE_API_KEY")
    if MOCK_JEV if MOCK_JEV is not None else (key is None):
        if not _mock_notice_shown:
            print("[MOCK JEV] No TYPESAFE_API_KEY - heuristic stand-in scores. "
                  "Probabilities are NOT calibrated model outputs; set the key for the real demo.")
            _mock_notice_shown = True
        return mock_jev_decisions(state, questions)

    qs = {}
    for qid, q in questions.items():
        entry = {"type": "noul", "instructions": q["instructions"]}
        if q.get("criteria"):
            entry["criteria"] = q["criteria"]          # {"true": ..., "false": ...} as-is
        qs[qid] = entry
    body = json.dumps({"model": JEV_MODEL, "state": state, "questions": qs}).encode()
    req = urllib.request.Request(JEV_ENDPOINT, data=body, headers={
        "Authorization": f"Bearer {key}", "Content-Type": "application/json"})
    last_err = None
    for attempt in range(retries + 1):
        try:
            with urllib.request.urlopen(req, timeout=timeout) as r:
                answers = json.loads(r.read())["answers"]
            return {qid: float(answers[qid]["noul"]) for qid in questions}
        except Exception as e:
            last_err = e
            time.sleep(2 ** attempt)
    raise last_err


# --- MOCK decision scores (offline demo stand-in, NOT a model) -------------------
# Deterministic heuristics over the state so the pipeline plumbing is testable with
# zero keys. Pass 1 sees the raw alert only; pass 2 sees retrieved telemetry - the
# mock rewards corroborating evidence, mimicking the real two-pass confidence shift.
_MOCK_BAD = ("-enc", "-nop", "mimikatz", "rubeus", ".sct", "mshta", "lsass", "minidump",
             "regsvr32 /s", "certutil -urlcache", "shadow", "kerberoast", "asreproast")
_MOCK_BENIGN = ("approved_change", "approved_security_test", "approved_pipeline",
                "approved_incident_response", "ticket_match", "approved_software_install",
                "scheduled_backup_window", "command_seen_before", "script_hash_known",
                "matches_approved_activity", "behavioral_baseline_match")

def mock_jev_decisions(state, questions=QUESTIONS):
    cmd = (state.get("trigger", {}).get("command_line") or "").lower()
    sev = state.get("alert", {}).get("severity", "low")
    risk = state.get("alert", {}).get("risk_score", 30) / 100.0
    net = state.get("network", {})
    enr = {**state.get("enrichment", {}), **state.get("correlated_activity", {})}
    has_telemetry = bool(state.get("enrichment") or state.get("correlated_activity"))

    bad = (sum(p in cmd for p in _MOCK_BAD)
           + (1 if net and not net.get("destination_internal", True)
              and not net.get("destination_seen_before", True) else 0)
           + (1 if enr.get("ti_or_behavioral_corroboration") or enr.get("malicious_toolchain_pattern") else 0))
    benign = sum(1 for k in _MOCK_BENIGN if enr.get(k))
    sev_w = {"critical": 0.25, "high": 0.15, "medium": 0.05, "low": -0.05}.get(sev, 0)

    tp = risk + sev_w + 0.12 * bad - 0.15 * benign
    if has_telemetry:                       # retrieved evidence sharpens the verdict
        tp = 0.5 + (tp - 0.5) * 1.8

    uncertain = 0.3 < tp < 0.85
    ext_indicator = bool(net and not net.get("destination_internal", True))
    clamp = lambda x: min(0.99, max(0.02, x))
    return {
        "true_positive": clamp(tp),
        "requires_immediate_response": clamp(tp + (0.08 if sev in ("high", "critical") else -0.10)),
        "requires_investigation_before_action": 0.75 if (uncertain and not has_telemetry)
                                                else (0.35 if uncertain else 0.08),
        "increase_monitoring": 0.70 if uncertain else (0.55 if tp >= 0.85 else 0.20),
        "preserve_evidence": clamp(tp + 0.05) if tp > 0.5 else 0.25,
        "block_indicator": clamp(tp + 0.03) if ext_indicator else 0.10,
        "isolate_endpoint": clamp(tp) if state.get("trigger", {}).get("process") else 0.10,
        "disable_account": clamp(tp - 0.02) if tp > 0.5 else 0.10,
    }

print(f"Decision engine: Jev ({JEV_MODEL})"
      + ("" if (TYPESAFE_API_KEY or os.environ.get("TYPESAFE_API_KEY")) and MOCK_JEV is not True
         else "  [MOCK MODE until TYPESAFE_API_KEY is set]"))

In [ ]:
# ============================================================
# 7. TRIAGE GATE — deterministic routing on pass-1 scores
# ============================================================
#   P(true_positive) <= 0.10                        -> AUTO_CLOSE (benign / FP)
#   P(true_positive) >= 0.95 and investigate < 0.35 -> AUTO_ACT
#   anything in between                             -> INVESTIGATE

def confidence_gate(p):
    if p["true_positive"] <= FP_CLOSE_THRESHOLD:
        return "AUTO_CLOSE"
    if p["true_positive"] >= TP_ACT_THRESHOLD and p["requires_investigation_before_action"] < INVESTIGATE_THRESHOLD:
        return "AUTO_ACT"
    return "INVESTIGATE"


def gate_reason(scores, route):
    """One-line 'why', with the actual numbers, for the trace."""
    tp, inv = scores["true_positive"], scores["requires_investigation_before_action"]
    if route == "AUTO_CLOSE":
        return f"P(true positive) {tp:.2f} <= {FP_CLOSE_THRESHOLD} -> benign / detection false positive"
    if route == "AUTO_ACT":
        return (f"P(true positive) {tp:.2f} >= {TP_ACT_THRESHOLD} and "
                f"P(needs investigation) {inv:.2f} < {INVESTIGATE_THRESHOLD} -> confident enough to act")
    return (f"P(true positive) {tp:.2f} but P(needs investigation) {inv:.2f} "
            f">= {INVESTIGATE_THRESHOLD} -> not enough confidence to auto-act")

In [ ]:
# ============================================================
# 8. TELEMETRY TOOLS — mock connectors (demo)
# ============================================================
# Each function is one connector you would replace with a real call:
# EDR (process tree, isolation), SIEM search, threat intel, IdP logs, CMDB.
# Demo convention: the mock "backends" know the ground truth stored in each
# demo state's enrichment section, and surface correlated_activity if present.

def get_process_tree(state, **kw):
    trg = state.get("trigger", {})
    suspicious_parent = trg.get("parent", "") in {"WINWORD.EXE", "OUTLOOK.EXE", "explorer.exe"}
    return {"process_tree": {
        "parent": trg.get("parent"), "process": trg.get("process"),
        "suspicious_parent_child": suspicious_parent,
        "parent_child_seen_before": state.get("enrichment", {}).get("parent_child_seen_before", not suspicious_parent)}}

def get_network_connections(state, **kw):
    net = state.get("network", {})
    return {"network": {
        "destination": net.get("destination") or state.get("trigger", {}).get("destination"),
        "destination_seen_before": net.get("destination_seen_before", True),
        "destination_internal": net.get("destination_internal", True),
        "tls_ja3_known_good": net.get("destination_internal", True)}}

def get_threat_intel(state, **kw):
    net = state.get("network", {})
    dest = net.get("destination") or state.get("trigger", {}).get("destination") or ""
    bad = (not net.get("destination_internal", True)) and (not net.get("destination_seen_before", True))
    return {"threat_intel": {
        "indicator": dest or "n/a",
        "ti_verdict": "malicious" if bad else "clean_or_unknown",
        "blocklist_hits": 3 if bad else 0}}

def get_auth_log(state, **kw):
    enr = state.get("enrichment", {})
    return {"auth_log": {
        "mfa_success": enr.get("mfa_success"),
        "source_seen_for_user_before": enr.get("source_seen_for_user_before", True),
        "impossible_travel": enr.get("impossible_travel", False),
        "failed_logons_1h": 42 if enr.get("mfa_success") is False else 0}}

def get_related_alerts(state, **kw):
    corr = state.get("correlated_activity", {})
    n = len(corr) if corr else state.get("enrichment", {}).get("similar_alerts_host_24h", 0)
    return {"related_alerts": {"count_24h": n, "details": corr or "none"}}

def get_asset_context(state, **kw):
    enr = state.get("enrichment", {})
    return {"asset_context": {
        "approved_change": enr.get("approved_change", False),
        "ticket_match": enr.get("ticket_match", False),
        "maintenance_window_active": enr.get("maintenance_window_active", False),
        "host_criticality": state.get("host", {}).get("criticality")}}

TOOL_REGISTRY = {
    "get_process_tree":         get_process_tree,
    "get_network_connections":  get_network_connections,
    "get_threat_intel":         get_threat_intel,
    "get_auth_log":             get_auth_log,
    "get_related_alerts":       get_related_alerts,
    "get_asset_context":        get_asset_context,
}
print("Telemetry tools:", ", ".join(TOOL_REGISTRY))

In [ ]:
# ============================================================
# 9. INVESTIGATOR — LLM (plan -> retrieve -> summarize -> explain)
# ============================================================
# Provider-agnostic: set INVESTIGATOR_PROVIDER in the config cell.
#   "gemini"            Google Gemini via the google-genai SDK
#   "openai_compatible" OpenAI, Minimax, or any /chat/completions endpoint (plain REST)
#   "claude"            Anthropic Claude via the Messages API (plain REST)
# The investigator sees only the RAW alert (same view as pass 1). It plans which
# telemetry tools to call, summarizes the results into structured evidence fields,
# and later writes the plain-English assessment. Guardrail: it may only append to
# the whitelisted telemetry sections of the state.

def _post_json(url, body, headers, timeout=120):
    req = urllib.request.Request(url, data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json", **headers})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())


def _with_retries(fn, retries=4, label="llm"):
    """Retry transient 429/503-style failures with exponential backoff."""
    last_err = None
    for attempt in range(retries + 1):
        try:
            return fn()
        except Exception as e:
            last_err = e
            transient = any(c in str(e) for c in ("503", "429", "UNAVAILABLE", "RESOURCE_EXHAUSTED", "502"))
            if not transient or attempt == retries:
                raise
            wait = min(2 ** attempt * 2, 30)
            print(f"  [{label} busy: retry {attempt + 1}/{retries} in {wait}s]")
            time.sleep(wait)
    raise last_err


# --- provider: Google Gemini ----------------------------------------------------
_gemini_client = None

def _gemini_client_get():
    global _gemini_client
    if _gemini_client is None:
        key = GEMINI_API_KEY or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
        from google import genai
        _gemini_client = genai.Client(api_key=key)
    return _gemini_client

def _gemini_chat(prompt, json_mode):
    config = {"response_mime_type": "application/json"} if json_mode else {}
    return _with_retries(lambda: _gemini_client_get().models.generate_content(
        model=GEMINI_MODEL, contents=prompt, config=config).text, label="gemini")


# --- provider: OpenAI-compatible (OpenAI / Minimax / ...) -----------------------
def _compat_chat(prompt, json_mode):
    key = COMPAT_API_KEY or os.environ.get("OPENAI_API_KEY") or os.environ.get("MINIMAX_API_KEY")
    if json_mode:   # JSON instruction in-prompt keeps every compatible endpoint happy
        prompt += "\n\nRespond with a single JSON object only, no prose, no markdown fences."
    def call():
        resp = _post_json(f"{COMPAT_BASE_URL}/chat/completions",
                          {"model": COMPAT_MODEL,
                           "messages": [{"role": "user", "content": prompt}]},
                          {"Authorization": f"Bearer {key}"})
        return resp["choices"][0]["message"]["content"]
    return _with_retries(call, label="compat")


# --- provider: Anthropic Claude --------------------------------------------------
def _claude_chat(prompt, json_mode):
    key = CLAUDE_API_KEY or os.environ.get("ANTHROPIC_API_KEY")
    if json_mode:
        prompt += "\n\nRespond with a single JSON object only, no prose, no markdown fences."
    def call():
        resp = _post_json("https://api.anthropic.com/v1/messages",
                          {"model": CLAUDE_MODEL, "max_tokens": 2048,
                           "messages": [{"role": "user", "content": prompt}]},
                          {"x-api-key": key, "anthropic-version": "2023-06-01"})
        return resp["content"][0]["text"]
    return _with_retries(call, label="claude")


# --- dispatcher ------------------------------------------------------------------
_PROVIDERS = {"gemini": _gemini_chat, "openai_compatible": _compat_chat, "claude": _claude_chat}
_PROVIDER_MODELS = {"gemini": GEMINI_MODEL, "openai_compatible": COMPAT_MODEL, "claude": CLAUDE_MODEL}

def llm_name():
    return _PROVIDER_MODELS[INVESTIGATOR_PROVIDER]

def llm_available():
    if INVESTIGATOR_PROVIDER == "gemini":
        if not (GEMINI_API_KEY or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")):
            return False
        try:
            import google.genai  # noqa: F401
            return True
        except ImportError:
            return False
    if INVESTIGATOR_PROVIDER == "openai_compatible":
        return bool(COMPAT_API_KEY or os.environ.get("OPENAI_API_KEY") or os.environ.get("MINIMAX_API_KEY"))
    if INVESTIGATOR_PROVIDER == "claude":
        return bool(CLAUDE_API_KEY or os.environ.get("ANTHROPIC_API_KEY"))
    return False

def llm_chat(prompt, json_mode=True):
    """One investigator call, routed to the configured provider."""
    return _PROVIDERS[INVESTIGATOR_PROVIDER](prompt, json_mode)


def extract_json(text):
    """Pull the first JSON object out of a response, tolerating prose around it."""
    m = re.search(r"\{.*\}", text, re.S)
    return json.loads(m.group()) if m else {}

In [ ]:
# --- Investigator prompts -----------------------------------------------------------

PLAN_PROMPT = """You are a SOC investigation planner. A fast classifier scored this alert:
PASS1_SCORES
Alert:
ALERT_JSON
Available tools: TOOL_LIST
Decide which telemetry to retrieve to confirm or refute this alert.
Respond ONLY with JSON: {"tool_calls": [{"tool": "<name>", "args": {}}], "hypothesis": "<one line>"}"""

SUMMARY_PROMPT = """You are a SOC investigator. Alert:
ALERT_JSON
Fast classifier scores: PASS1_SCORES
Retrieved telemetry:
TOOL_RESULTS
Summarize the evidence as structured fields to append to the alert state.
Respond ONLY with JSON of the shape:
{"correlated_activity": {...}, "enrichment": {...}}
Use short snake_case boolean/number/string fields. Only include fields supported by the telemetry."""

EXPLAIN_PROMPT = """You are a senior SOC analyst writing the assessment line of an incident ticket.
Alert (raw): ALERT_JSON
Pass-1 machine scores (raw alert only): PASS1
Telemetry retrieved during investigation: EVIDENCE
Pass-2 machine scores (with retrieved evidence): PASS2
Actions auto-executed: AUTO_ACTIONS
Actions held for human approval: HUMAN_ACTIONS
Disposition: DISPOSITION
Write 3-5 sentences of plain English for a human analyst: what this alert is, what the
retrieved evidence showed, why the confidence moved between passes (cite the numbers),
and why each action was taken or held. Be concrete; no jargon filler; no bullet lists."""


def llm_investigate(state, p1):
    """Plan -> retrieve -> summarize. Returns (evidence_fields, trace)."""
    alert_json = json.dumps(strip_for_pass1(state), indent=1)   # raw view only
    scores = json.dumps({k: round(v, 3) for k, v in p1.items()})

    plan = extract_json(llm_chat(PLAN_PROMPT
                                 .replace("PASS1_SCORES", scores)
                                 .replace("ALERT_JSON", alert_json)
                                 .replace("TOOL_LIST", ", ".join(TOOL_REGISTRY))))
    calls = ([c for c in plan.get("tool_calls", []) if c.get("tool") in TOOL_REGISTRY]
             or [{"tool": t, "args": {}} for t in TOOL_REGISTRY])   # fallback: pull everything

    results = {}
    for c in calls:
        results.update(TOOL_REGISTRY[c["tool"]](state, **c.get("args", {})))

    extra = extract_json(llm_chat(SUMMARY_PROMPT
                                  .replace("ALERT_JSON", alert_json)
                                  .replace("PASS1_SCORES", scores)
                                  .replace("TOOL_RESULTS", json.dumps(results, indent=1))))
    trace = {"investigator": llm_name(), "hypothesis": plan.get("hypothesis", ""),
             "tools_called": [c["tool"] for c in calls], "raw_results": results}
    return sanitize_extra(extra), trace


def llm_explain(context):
    return llm_chat(EXPLAIN_PROMPT
                    .replace("ALERT_JSON", json.dumps(context["raw"], indent=1))
                    .replace("PASS1", json.dumps({k: round(v, 2) for k, v in context["p1"].items()}))
                    .replace("EVIDENCE", json.dumps(context.get("evidence", {}), indent=1))
                    .replace("PASS2", json.dumps({k: round(v, 2) for k, v in context["scores"].items()}))
                    .replace("AUTO_ACTIONS", json.dumps([d for _, d in context["executed"]]))
                    .replace("HUMAN_ACTIONS", json.dumps([d for _, d in context["queued"]]))
                    .replace("DISPOSITION", context["disposition"]),
                    json_mode=False).strip()

In [ ]:
# ============================================================
# 10. INVESTIGATOR — heuristic fallback + dispatchers
# ============================================================
# Runs with no API key. In this demo the "telemetry backends" are the demo states
# themselves: investigating = surfacing the enrichment / correlated_activity
# sections that pass 1 was not allowed to see, plus corroboration derived from
# the raw alert. The configured LLM slots in transparently when its key is present.

MALWARE_PATTERNS = ("-enc", "-nop", "mimikatz", "rubeus", ".sct", "mshta", "lsass",
                    "minidump", "regsvr32 /s", "certutil -urlcache", "shadow")

def heuristic_investigate(state, p1):
    raw = strip_for_pass1(state)
    cmd = (raw.get("trigger", {}).get("command_line") or "").lower()
    net = raw.get("network", {})

    # Retrieved telemetry = the sections withheld from pass 1
    extra = {}
    for key in HIDDEN_UNTIL_INVESTIGATION:
        if isinstance(state.get(key), dict) and state[key]:
            extra[key] = deepcopy(state[key])

    enr = extra.get("enrichment", {})
    benign = any(enr.get(k) for k in
                 ("approved_change", "approved_security_test", "approved_pipeline",
                  "approved_incident_response", "ticket_match", "approved_software_install",
                  "scheduled_backup_window", "command_seen_before", "script_hash_known"))
    bad = (any(p in cmd for p in MALWARE_PATTERNS)
           or (net and not net.get("destination_internal", True)
               and not net.get("destination_seen_before", True)))
    if bad and not benign:
        extra.setdefault("correlated_activity", {}).update(
            {"malicious_toolchain_pattern": True, "ti_or_behavioral_corroboration": True})
    elif benign:
        extra.setdefault("correlated_activity", {}).update(
            {"matches_approved_activity": True, "behavioral_baseline_match": True})

    hyp = "malicious" if bad and not benign else ("benign/expected" if benign else "uncertain")
    trace = {"investigator": "heuristic-fallback", "hypothesis": hyp,
             "tools_called": list(TOOL_REGISTRY),
             "raw_results": "(retrieved withheld sections + derived flags)"}
    return extra, trace


def heuristic_explain(context):
    """Template-based assessment when Gemini is unavailable."""
    p1, s2 = context["p1"], context["scores"]
    a = context["raw"].get("alert", {})
    inv = context.get("investigator", {})
    parts = [
        f"{context['case_id']} was raised by the '{a.get('rule')}' rule "
        f"(severity {a.get('severity')}, risk {a.get('risk_score')}).",
        f"On the raw alert, Jev scored P(true positive)={p1['true_positive']:.2f} with "
        f"P(needs investigation)={p1['requires_investigation_before_action']:.2f}.",
    ]
    if context.get("pass2"):
        parts.append(
            f"The {inv.get('investigator', 'investigator')} ({inv.get('hypothesis', 'no hypothesis')}) "
            f"retrieved {', '.join(context.get('evidence_sections', [])) or 'no additional'} telemetry, "
            f"moving P(true positive) to {s2['true_positive']:.2f} and "
            f"P(needs investigation) to {s2['requires_investigation_before_action']:.2f}.")
    if context["executed"]:
        parts.append("Auto-executed: " + "; ".join(d for _, d in context["executed"]) + ".")
    if context["queued"]:
        parts.append("Held for human approval: " + "; ".join(d for _, d in context["queued"]) + ".")
    if not context["executed"] and not context["queued"]:
        parts.append("No response actions met their thresholds.")
    parts.append(f"Disposition: {context['disposition']}.")
    return " ".join(parts)


# --- Dispatchers: Gemini when available, heuristic otherwise --------------------

def sanitize_extra(extra):
    """Guardrail: the investigator may only append to the whitelisted state sections."""
    return {k: v for k, v in extra.items()
            if k in HIDDEN_UNTIL_INVESTIGATION and isinstance(v, dict)}

def investigate(state, p1):
    if llm_available():
        try:
            return llm_investigate(state, p1)
        except Exception as e:
            print(f"  [{INVESTIGATOR_PROVIDER} failed: {e}] falling back to heuristic investigator")
    return heuristic_investigate(state, p1)

def explain_outcome(context):
    if llm_available():
        try:
            return llm_explain(context)
        except Exception as e:
            print(f"  [{INVESTIGATOR_PROVIDER} explain failed: {e}] using template summary")
    return heuristic_explain(context)

In [ ]:
# ============================================================
# 11. RESPONSE POLICY — deterministic mapping from scores to actions
# ============================================================
# Three safeguards on top of the per-action thresholds:
#   1. Consensus: an action only fires if P(true_positive) clears the global bar
#      AND P(needs investigation) is low.
#   2. Impact-tiered thresholds: reversible actions auto-fire low;
#      isolate / disable need >= 0.95.
#   3. High-impact actions on critical assets always queue for human sign-off.
# Handler bodies are stubs - wire them to firewall / EDR / IdP / case management.

def _asset_label(state):
    """Best-effort asset name - cloud/SaaS/email alerts may have no host section."""
    return (state.get("host", {}).get("name")
            or state.get("cloud", {}).get("resource")
            or state.get("cloud", {}).get("provider")
            or "affected asset")


def _user_label(state):
    return state.get("user", {}).get("name", "affected account")


def do_increase_monitoring(state): return f"Enable enhanced telemetry on {_asset_label(state)}"
def do_preserve_evidence(state):   return f"Snapshot memory+disk, retain logs for {_asset_label(state)}"
def do_block_indicator(state):
    ind = (state.get("network", {}).get("destination")
           or state.get("trigger", {}).get("destination") or "indicator")
    return f"Push block for {ind} to egress firewall / DNS RPZ"
def do_isolate_endpoint(state):    return f"EDR network-isolate {_asset_label(state)}"
def do_disable_account(state):     return f"Disable {_user_label(state)} in IdP + revoke sessions"

ACTION_HANDLERS = {
    "increase_monitoring": do_increase_monitoring,
    "preserve_evidence":   do_preserve_evidence,
    "block_indicator":     do_block_indicator,
    "isolate_endpoint":    do_isolate_endpoint,
    "disable_account":     do_disable_account,
}


def hold_reason(qid, state):
    if (ACTION_POLICY[qid]["impact"] == "high"
            and state.get("host", {}).get("criticality") in CRITICAL_HOST_LEVELS
            and HUMAN_APPROVAL_ON_CRITICAL):
        return "critical asset - sign-off required"
    return "below auto-act consensus"


def action_policy(state, scores):
    """Returns (executed, queued_for_human)."""
    consensus = (scores["true_positive"] >= TP_ACT_THRESHOLD
                 and scores["requires_investigation_before_action"] < INVESTIGATE_THRESHOLD)
    executed, queued = [], []
    for qid, pol in ACTION_POLICY.items():
        if scores.get(qid, 0) < pol["threshold"]:
            continue
        desc = ACTION_HANDLERS[qid](state)
        (executed if consensus and hold_reason(qid, state) == "below auto-act consensus"
         else queued).append((qid, desc))
    return executed, queued

In [ ]:
# ============================================================
# 12. PIPELINE — one alert, end to end
# ============================================================

def process_alert(state, verbose=True):
    trace = {"case_id": state.get("case_id"), "route": None}
    context = {"case_id": state.get("case_id"), "raw": strip_for_pass1(state)}

    # --- PASS 1: raw alert only (no withheld telemetry) -----------------------
    raw = strip_for_pass1(state)
    t0 = time.time()
    p1 = jev_decisions(raw)
    trace["pass1"], trace["pass1_ms"] = p1, round((time.time() - t0) * 1000)
    route = confidence_gate(p1)
    trace["route"] = route
    if verbose:
        a = state["alert"]
        h, u = state.get("host", {}), state.get("user", {})
        print(f"\n{'=' * 74}")
        print(f"{state['case_id']} — {a['rule']}")
        print(f"host: {h.get('name', 'n/a')} ({h.get('role', '?')}, {h.get('criticality', '?')})   "
              f"user: {u.get('name', 'n/a')}   severity: {a['severity']} (risk {a['risk_score']})")
        print(f"\nPASS 1 — Jev scores the raw alert  [{trace['pass1_ms']} ms]")
        fmt_scores(p1)
        print(f">> {gate_reason(p1, route)}")

    # --- INVESTIGATE: retrieve telemetry, then PASS 2 on the enriched state ---
    enriched = deepcopy(raw)        # pass-2 state = raw alert + retrieved evidence ONLY
    if route == "INVESTIGATE":
        extra, inv_trace = investigate(state, p1)
        trace["investigator"] = inv_trace
        context.update(investigator=inv_trace, evidence=extra, evidence_sections=list(extra))
        enriched = deep_merge(enriched, extra)
        if verbose:
            print(f"\nINVESTIGATION — {inv_trace['investigator']}")
            print(f"    hypothesis: {inv_trace['hypothesis']}")
            print(f"    telemetry pulled: {', '.join(inv_trace['tools_called'])}")
            print(f"    evidence added to state: {', '.join(extra) or 'none'}")

        t0 = time.time()
        p2 = jev_decisions(enriched)
        trace["pass2"], trace["pass2_ms"] = p2, round((time.time() - t0) * 1000)
        if verbose:
            print(f"\nPASS 2 — Jev re-scores with the retrieved evidence  [{trace['pass2_ms']} ms]")
            fmt_scores(p2, prev=p1)
            print(f">> {gate_reason(p2, confidence_gate(p2))}")
        scores = p2
        context["pass2"] = p2
        if confidence_gate(p2) == "AUTO_CLOSE":
            trace["route"] = "AUTO_CLOSE_AFTER_INVESTIGATION"
    else:
        scores = p1

    # --- DECIDE ----------------------------------------------------------------
    context["p1"], context["scores"] = p1, scores
    if trace["route"].startswith("AUTO_CLOSE"):
        trace["executed"], trace["queued"] = [], []
        trace["disposition"] = "closed_benign"
        if verbose:
            print("\nDISPOSITION: CLOSED — no response actions warranted")
    else:
        executed, queued = action_policy(enriched, scores)
        trace["executed"], trace["queued"] = executed, queued
        trace["disposition"] = ("human_required"
                                if (queued and not executed) or confidence_gate(scores) == "INVESTIGATE"
                                else "auto_responded")
        if verbose:
            print("\nRESPONSE")
            for qid, desc in executed:
                print(f"  [AUTO]   {qid.replace('_', ' ')}: {desc}")
            for qid, desc in queued:
                print(f"  [HUMAN]  {qid.replace('_', ' ')}: {desc}  ({hold_reason(qid, enriched)})")
            if not executed and not queued:
                print("  no action thresholds met -> queued for human triage")

    # --- ASSESSMENT: plain-English why, from the investigator ------------------
    context.update(executed=trace["executed"], queued=trace["queued"],
                   disposition=trace["disposition"])
    trace["explanation"] = explain_outcome(context)
    if verbose:
        print("\nASSESSMENT")
        print(textwrap.indent(textwrap.fill(trace["explanation"], width=72), "  "))
    return trace

## Run the demo

Three cases that show the three routes. Watch the `>>` line after each pass (why the gate
routed it), what the investigator retrieves, how pass 2 moves vs pass 1, and which actions
auto-fire vs queue for a human.

In [ ]:
# --- Three representative cases --------------------------------------------------
DEMO_NOTES = {
    "VIRGIL-001": "approved admin PowerShell - looks bad raw, should clear after investigation",
    "VIRGIL-009": "ambiguous user PowerShell bypass - expect investigation, then a verdict",
    "VIRGIL-034": "ransomware on a critical file server - expect high-confidence response",
}
for cid, note in DEMO_NOTES.items():
    print(f"\n### {cid} — {note}")
    _trace = process_alert(next(x for x in STATES if x["case_id"] == cid))

## Run any alert

In [ ]:
# ============================================================
# RUN A SINGLE ALERT — choose any state from the dropdown
# ============================================================
try:
    import ipywidgets as widgets

    _options = [(f"{s['case_id']} — {s['alert']['rule']} "
                 f"({s['alert']['severity']}, {s['host']['name']})", s["case_id"]) for s in STATES]
    _picker = widgets.Dropdown(options=_options, description="Alert:",
                               layout=widgets.Layout(width="85%"),
                               style={"description_width": "60px"})
    _btn = widgets.Button(description="Run pipeline", button_style="primary",
                          layout=widgets.Layout(width="160px"))
    _out = widgets.Output()

    def _run_clicked(_):
        with _out:
            _out.clear_output()
            state = next(x for x in STATES if x["case_id"] == _picker.value)
            process_alert(state)

    _btn.on_click(_run_clicked)
    display(widgets.VBox([_picker, _btn, _out]))

except ImportError:
    # No widgets available: set CASE_ID and re-run this cell instead.
    CASE_ID = "VIRGIL-002"
    process_alert(next(x for x in STATES if x["case_id"] == CASE_ID))